# Logistic Regression Baseline: Production vs. Changeover (`Label_03`)

**Project:** Machine-learning-based recognition of production and changeover activities  
**CRISP-DM phase:** Modelling / Evaluation

## Purpose

This notebook trains a simple, interpretable **baseline** classifier: logistic regression predicting the binary label `Label_03` (**1 = Production, 0 = Changeover**) from 40 selected machine signals of a five-axis CNC milling machine (Spinner U5-620 with Siemens 840D-SL control). More complex models can be compared against this reference. The baseline is described in the accompanying bachelor's thesis (Sections 3.2 and 4.1).

It:

1. loads the min-max-scaled dataset produced by notebook 01;
2. selects the 40 features listed below;
3. splits the data into training (80 %) and test (20 %) sets;
4. trains a pipeline of median imputation, standardization and logistic regression;
5. evaluates it (AUC, accuracy, precision, recall, F1, confusion matrix, ROC curve);
6. inspects the largest standardized coefficients;
7. saves the results locally.

## Workflow

Run notebook `01_Data_Preparation_and_Cleaning_GitHub.ipynb` first. It creates `normalized_data.csv` in the project root, which this notebook reads from `../normalized_data.csv`.

> **Data:** The dataset is public (Zenodo, licence CC BY 4.0) and is **not** included in this repository. See the project README for the download link and citation; notebook 01 turns it into `normalized_data.csv`. The files written to `../outputs/` are generated locally and are not committed to the repository.

## Important limitations (read before interpreting results)

- **Random split on time-series data.** The split below is a stratified *random* split of rows. Neighbouring time steps of a machine are very similar, so near-duplicates can end up in both training and test sets and the scores can be **optimistic**. A chronological or group-wise split (e.g. by production session) is a stricter test; the time-series random forest in the thesis uses chronological splitting for this reason.
- **Origin of the 40 features.** The 40 features are the 20 most negative and the 20 most positive Pearson correlations with `Label_03`, taken from the correlation ranking in notebook 01 (Section 8; thesis Figure 2). That ranking was computed on the *whole* dataset, including the rows that later end up in the test set, so the test scores are slightly biased. A stricter design selects features on the training data only.
- **Class coding.** Class `1` = Production is the positive class and class `0` = Changeover. Precision and recall below therefore describe the Production class.

## 1. Environment and configuration

Install the dependencies with `pip install -r requirements.txt`. All paths are relative to this notebook, which is expected to live in the `notebooks/` folder.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    RocCurveDisplay, accuracy_score, confusion_matrix,
    f1_score, precision_score, recall_score, roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

DATA_PATH = Path("../normalized_data.csv")   # created by notebook 01
OUTPUT_DIR = Path("../outputs")              # local results, not for upload

LABEL_COL = "Label_03"
SEED = 42            # makes the split and the model reproducible
TEST_SIZE = 0.2      # 20 % of the rows are held out for testing
THRESHOLD = 0.5      # probability above which a row is predicted as class 1

## 2. Load the normalized data

The file `normalized_data.csv` comes from notebook 01. The notebook stops with a clear message if the file or the target column is missing.

In [ ]:
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Could not find {DATA_PATH.resolve()}. "
        "Run notebook 01 first (it creates normalized_data.csv in the project root) "
        "or update DATA_PATH."
    )

df = pd.read_csv(DATA_PATH)
print(f"Loaded: {DATA_PATH}")
print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")

if LABEL_COL not in df.columns:
    raise KeyError(
        f"Target column {LABEL_COL!r} not found. "
        f"The file has {len(df.columns)} columns."
    )

## 3. Select the 40 features

The model uses a fixed list of 40 machine signals (spindle, axes, tool magazine, door, program state, motor temperatures, ...), chosen from the correlation ranking of notebook 01 (see the limitations above). Columns that are missing from the file are reported; the model then continues with the columns that are present, so check the printed count.

In [ ]:
feature_list = [
    'Spindle_smoothed_speed',
    'Program_status',
    'Z-axis_position_WCS',
    'Magazine_9998_number_Place_2',
    'Door_open',
    'Z-axis_position_MCS',
    'Magazine_9998_position_Place_3',
    'Program_Idle',
    'Magazine_position_Place_9',
    'Tool_number_Magazine_Place_43',
    'Magazine_number_Place_6',
    'Magazine_number_Place_7',
    'Magazine_number_Place_5',
    'Magazine_9998_position_Place_1',
    'Tool_number_Magazine_Place_9',
    'Magazine_number_Place_8',
    'Tool_magazine_position_WCS',
    'Tool_magazine_position_MCS',
    'Magazine_position_Place_17',
    'Number_of_tools',
    'Feed_override',
    'Rapid_traverse_button',
    'C_Axis_Motor_temperature',
    'Y_Axis_Motor_temperature',
    'Door_status_locked',
    'Spindle_position_WCS',
    'Spindle_position_MCS',
    'X-axis_position_MCS',
    'Net_running_time_program',
    'X_Axis_motor_temperature',
    'B_Axis_Motor_temperature',
    'Program_line_number',
    'Tool_number_Magazine_9998_Place_1',
    'X-axis_position_WCS',
    'Operation_mode',
    'LED_Chip_flushing',
    'Spindle_angular_velocity',
    'Spindle_speed',
    'Spindle_smoothed_modulation_depth',
    'LED_Coolant',
]

missing_features = [c for c in feature_list if c not in df.columns]
if missing_features:
    print("Warning: these features are missing and will be skipped:", missing_features)

feature_list = [c for c in feature_list if c in df.columns]
if not feature_list:
    raise ValueError("None of the expected features were found in the data.")
print(f"Using {len(feature_list)} features")

## 4. Build the feature matrix and split the data

`X` holds the features and `y` the target. Infinite values (if any) are turned into missing values; the missing values themselves are filled later **inside the model pipeline**, using medians learned from the training rows only. The split is stratified, so both sets keep the same share of each class.

Reminder: this is a random split (see the limitations above).

In [ ]:
X = df[feature_list].replace([np.inf, -np.inf], np.nan)
y = df[LABEL_COL].astype(int).to_numpy()

if set(np.unique(y)) != {0, 1}:
    raise ValueError(
        f"{LABEL_COL} must contain exactly the classes 0 and 1, "
        f"found: {np.unique(y).tolist()}"
    )

print(f"Missing / infinite feature values: {int(X.isna().sum().sum()):,}")
print(f"Share of class 1: {y.mean():.1%}")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=SEED, stratify=y
)
print(f"Train: {X_train.shape}   Test: {X_test.shape}")

## 5. Train the baseline model

The pipeline applies three steps, each fitted on the training data only:

1. **Median imputation** for any missing values;
2. **Standardization** (mean 0, standard deviation 1), so the coefficients are comparable;
3. **Logistic regression** with L2 regularization (the default) and `class_weight="balanced"`, which compensates for the unequal class sizes.

The data is already min-max-scaled by notebook 01. Standardizing it again does not change the result (it is a linear rescaling) but keeps this notebook correct if it is used with unscaled data.

In [ ]:
pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("logreg", LogisticRegression(
        solver="liblinear",
        class_weight="balanced",
        random_state=SEED,
        max_iter=1000,
    )),
])

pipe.fit(X_train, y_train)
print("Model trained.")

## 6. Evaluate on the test set

The model outputs a probability for class 1; rows with a probability of at least `THRESHOLD` (0.5) are predicted as class 1.

- **AUC:** how well the model ranks class-1 rows above class-0 rows (1.0 = perfect, 0.5 = random).
- **Precision:** of the rows predicted as class 1, the share that really are class 1.
- **Recall:** of all real class-1 rows, the share the model finds.
- **F1-score:** the balance of precision and recall.

In [ ]:
proba = pipe.predict_proba(X_test)[:, 1]
pred = (proba >= THRESHOLD).astype(int)

metrics = {
    "AUC": roc_auc_score(y_test, proba),
    "Accuracy": accuracy_score(y_test, pred),
    "F1-Score": f1_score(y_test, pred),
    "Precision": precision_score(y_test, pred, zero_division=0),
    "Recall": recall_score(y_test, pred),
}
display(pd.Series(metrics, name="test_score").round(4).to_frame())

cm = confusion_matrix(y_test, pred)
cm_df = pd.DataFrame(cm, index=["True 0", "True 1"], columns=["Pred 0", "Pred 1"])
print("Confusion matrix:")
display(cm_df)

RocCurveDisplay.from_predictions(y_test, proba)
plt.title("ROC curve: Logistic Regression baseline")
plt.tight_layout()
plt.show()

plt.figure(figsize=(4, 3))
sns.heatmap(cm_df, annot=True, fmt="d", cmap="Blues")
plt.title("Confusion matrix: LR baseline")
plt.tight_layout()
plt.show()

## 7. Inspect the model coefficients

Because the features are standardized, the size of a coefficient shows how strongly a feature moves the prediction: positive values push towards class 1, negative values towards class 0.

**Interpret with care:** many features are strongly related to each other (for example the motor temperatures or the tool-magazine places). With correlated features, individual coefficients can be unstable, so treat this ranking as a hint, not as proof of importance or causation.

In [ ]:
coefficients = pipe.named_steps["logreg"].coef_.ravel()

coef_df = pd.DataFrame({
    "feature": X.columns,
    "coef": coefficients,
    "abs_coef": np.abs(coefficients),
}).sort_values("abs_coef", ascending=False)

top20 = coef_df.head(20)
display(top20[["feature", "coef"]].reset_index(drop=True))

plt.figure(figsize=(8, 6))
plt.barh(top20["feature"][::-1], top20["coef"][::-1])   # largest at the top
plt.title("Top 20 standardized coefficients: LR baseline")
plt.xlabel("Coefficient")
plt.tight_layout()
plt.show()

## 8. Save the results locally

Three files are written to `../outputs/`:

- `label03_top40_features_used.csv`: the features used;
- `logreg_coefficients.csv`: all coefficients;
- `logreg_test_predictions.csv`: true label, predicted probability and prediction for every test row.

> **Do not upload these files.** They are derived from the private dataset (the predictions are row-level). The folder `outputs/` should be listed in `.gitignore`.

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

features_out = OUTPUT_DIR / "label03_top40_features_used.csv"
coefs_out = OUTPUT_DIR / "logreg_coefficients.csv"
preds_out = OUTPUT_DIR / "logreg_test_predictions.csv"

pd.Series(X.columns, name="feature").to_csv(features_out, index=False)
coef_df.to_csv(coefs_out, index=False)
pd.DataFrame({"y_true": y_test, "y_proba": proba, "y_pred": pred}).to_csv(
    preds_out, index=False
)

for path in (features_out, coefs_out, preds_out):
    print("Saved:", path)

## 9. Summary and conclusion

This notebook provides a transparent logistic-regression baseline for separating Production (1) from Changeover (0) (`Label_03`) with 40 machine signals. The pipeline fits imputation and scaling on the training data only, and the evaluation reports threshold-based metrics, AUC, a confusion matrix and the largest coefficients.

**Reference results from the author's original run** (the same model is reported in the thesis, Table 2, which differs marginally in the fourth decimal; the cleaned notebook may give slightly different numbers when re-run):

| Metric | Test value |
|---|---|
| AUC | 0.9684 |
| Accuracy | 0.9138 |
| F1-score | 0.8666 |
| Precision | 0.8115 |
| Recall | 0.9298 |

Test set: 10,406 rows. On this test set the model finds most class-1 rows (high recall) at the cost of a noticeable number of false alarms (lower precision).

**Next steps:** compare this baseline with the tree-based models studied in the thesis (decision tree, time-series random forest); replace the random split by a chronological or group-wise split to check how much the scores drop; select the features on training data only.